In [ ]:
!pip install -q pandas requests tqdm scikit-learn

In [ ]:
!pip install -q bitsandbytes accelerate

In [ ]:
import os
import sys
import time
import json
import warnings
import logging
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Silence the repeated "MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16"
# warning emitted by bitsandbytes on every forward pass during 8-bit quantized inference.
# It's expected/harmless given TigerLLM is natively bfloat16 and 8-bit matmul needs float16.
warnings.filterwarnings("ignore", message=".*MatMul8bitLt.*")
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)

# 1. KAGGLE INPUT PATH (Replace 'bsmdd-dataset' with your dataset folder name on Kaggle)
CSV_PATH = "/kaggle/input/datasets/syedmdnafissameen/depression/BSMDD_v3_2000_stratified.csv"

# Model
MODEL = "md-nishat-008/TigerLLM-9B-it"

# Number of shots PER CLASS (e.g. 2 -> 2 examples of label 0 + 2 examples of label 1 = 4-shot)
SHOTS_PER_CLASS = 2

SYSTEM_PROMPT = """Classify the given Bangla social media text into following categories:
1 = Depressive (hopelessness, suicidal, self-harm, clear depression).
0 = Non-depressive (stress/anger alone doesn't equal depression).
Output EXACTLY '0' or '1'. No explanations."""

USER_TEMPLATE = "Text: {text}\nLabel:"


def parse_label(raw_text):
    if raw_text is None:
        return None
    s = raw_text.strip()
    for ch in s:
        if ch in ["0", "০"]:
            return 0
        if ch in ["1", "১"]:
            return 1
    return None


def select_few_shot_examples(df, shots_per_class, seed=42):
    """Pick a fixed set of few-shot examples (balanced across classes) and
    return (few_shot_examples, remaining_df) so the shots are excluded from
    the evaluation set (no leakage)."""
    picked_indices = []
    for label_value in sorted(df["label"].unique()):
        subset = df[df["label"] == label_value].sample(
            n=shots_per_class, random_state=seed
        )
        picked_indices.extend(subset.index.tolist())

    few_shot_df = df.loc[picked_indices].sample(frac=1, random_state=seed)  # shuffle order
    few_shot_examples = few_shot_df[["text", "label"]].to_dict("records")

    remaining_df = df.drop(index=picked_indices).reset_index(drop=True)
    return few_shot_examples, remaining_df


def build_few_shot_messages(few_shot_examples, text):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for ex in few_shot_examples:
        messages.append({"role": "user", "content": USER_TEMPLATE.format(text=ex["text"])})
        messages.append({"role": "assistant", "content": str(ex["label"])})
    messages.append({"role": "user", "content": USER_TEMPLATE.format(text=text)})
    return messages


def call_model(model_obj, tokenizer_obj, few_shot_examples, text):
    messages = build_few_shot_messages(few_shot_examples, text)

    if hasattr(tokenizer_obj, "apply_chat_template"):
        prompt_input = tokenizer_obj.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        shots_str = ""
        for ex in few_shot_examples:
            shots_str += f"{USER_TEMPLATE.format(text=ex['text'])} {ex['label']}\n\n"
        prompt_input = f"{SYSTEM_PROMPT}\n\n{shots_str}{USER_TEMPLATE.format(text=text)}"

    start = time.monotonic()
    try:
        inputs = tokenizer_obj(prompt_input, return_tensors="pt").to(model_obj.device)
        with torch.no_grad():
            outputs = model_obj.generate(
                **inputs,
                max_new_tokens=5,
                do_sample=False  # Do not pass temperature when do_sample=False
            )
        content = tokenizer_obj.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        return content, None, time.monotonic() - start
    except Exception as e:
        return None, f"inference_error:{e}", time.monotonic() - start


def checkpoint_path(out_dir, model):
    col_name = model.replace("/", "__")
    return os.path.join(out_dir, f"checkpoint_fewshot_{col_name}.csv")


def load_checkpoint(out_dir, model):
    path = checkpoint_path(out_dir, model)
    if os.path.exists(path):
        return pd.read_csv(path)
    return None


def save_checkpoint(out_dir, model, rows):
    path = checkpoint_path(out_dir, model)
    pd.DataFrame(rows).to_csv(path, index=False)


def run_model_on_dataset(model_id, df, model_obj, tokenizer_obj, few_shot_examples, out_dir, checkpoint_every=20):
    existing = load_checkpoint(out_dir, model_id)
    rows = existing.to_dict("records") if existing is not None else []
    start_idx = len(rows)

    if start_idx >= len(df):
        return rows

    texts = df["text"].tolist()
    labels = df["label"].tolist()

    pbar = tqdm(range(start_idx, len(df)), desc=model_id, unit="row", initial=start_idx, total=len(df))
    for i in pbar:
        text = texts[i]
        label = labels[i]
        raw, err, latency = call_model(model_obj, tokenizer_obj, few_shot_examples, text)
        pred = parse_label(raw)
        correct = (pred == label) if pred is not None else False

        rows.append({
            "text": text,
            "label": label,
            "prediction": pred,
            "correct": correct,
            "latency": latency,
            "raw_response": raw,
            "error": err,
        })

        if (i - start_idx + 1) % checkpoint_every == 0:
            save_checkpoint(out_dir, model_id, rows)

    save_checkpoint(out_dir, model_id, rows)
    return rows


def compute_metrics(rows):
    valid = [r for r in rows if r["prediction"] is not None]
    if not valid:
        return {
            "accuracy": None, "precision": None, "recall": None,
            "f1": None, "confusion_matrix": None,
            "valid_predictions": 0, "total": len(rows),
            "avg_latency": None,
        }

    yt = [r["label"] for r in valid]
    yp = [r["prediction"] for r in valid]
    latencies = [r["latency"] for r in rows if r["latency"] is not None]

    acc = accuracy_score(yt, yp)
    prec = precision_score(yt, yp, zero_division=0)
    rec = recall_score(yt, yp, zero_division=0)
    f1 = f1_score(yt, yp, zero_division=0)
    cm = confusion_matrix(yt, yp, labels=[0, 1]).tolist()

    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "confusion_matrix": cm,
        "valid_predictions": len(valid),
        "total": len(rows),
        "avg_latency": sum(latencies) / len(latencies) if latencies else None,
    }


def main():
    # Configuration
    NUM_SAMPLES = None    # Set to None to evaluate all remaining samples (after removing few-shot examples)

    # 2. KAGGLE OUTPUT PATH
    OUTPUT_DIR = "/kaggle/working"
    CHECKPOINT_EVERY = 20

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    df = pd.read_csv(CSV_PATH)

    if "text" not in df.columns or "label" not in df.columns:
        raise ValueError("CSV must contain 'text' and 'label' columns")

    df["label"] = df["label"].astype(int)

    # Select fixed few-shot examples and remove them from the evaluation set (avoid leakage)
    few_shot_examples, df = select_few_shot_examples(df, SHOTS_PER_CLASS, seed=42)

    # Save the exact shots used, for reproducibility/reporting
    with open(os.path.join(OUTPUT_DIR, "few_shot_examples.json"), "w", encoding="utf-8") as f:
        json.dump(few_shot_examples, f, ensure_ascii=False, indent=2)

    if NUM_SAMPLES is not None:
        df = df.head(NUM_SAMPLES).reset_index(drop=True)
    else:
        df = df.reset_index(drop=True)

    # KEPT EXACTLY AS YOUR ORIGINAL CODE (8-Bit Quantization):
    quantization_config = BitsAndBytesConfig(
        load_in_8bit=True
    )

    print(f"Loading {MODEL} in 8-bit precision onto GPU...")
    tokenizer_obj = AutoTokenizer.from_pretrained(MODEL)
    model_obj = AutoModelForCausalLM.from_pretrained(
        MODEL,
        quantization_config=quantization_config,
        device_map="auto"
    )

    rows = run_model_on_dataset(
        MODEL,
        df,
        model_obj,
        tokenizer_obj,
        few_shot_examples,
        OUTPUT_DIR,
        CHECKPOINT_EVERY
    )

    pred_rows = []
    for r in rows:
        pred_rows.append({
            "text": r["text"],
            "label": r["label"],
            "prediction": r["prediction"],
            "correct": r["correct"],
            "latency": r["latency"],
            "raw_response": r["raw_response"],
            "error": r.get("error"),
        })

    metrics = compute_metrics(rows)

    results_row = {
        "model": MODEL,
        "shots_per_class": SHOTS_PER_CLASS,
        "total_shots": len(few_shot_examples),
        "accuracy": metrics["accuracy"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
        "confusion_matrix": json.dumps(metrics["confusion_matrix"]),
        "valid_predictions": metrics["valid_predictions"],
        "total": metrics["total"],
        "avg_latency": metrics["avg_latency"],
    }

    pred_path = os.path.join(OUTPUT_DIR, "predictions_fewshot.csv")
    pd.DataFrame(pred_rows).to_csv(pred_path, index=False, encoding="utf-8-sig")

    results_path = os.path.join(OUTPUT_DIR, "benchmark_results_fewshot.csv")
    pd.DataFrame([results_row]).to_csv(
        results_path,
        index=False,
        encoding="utf-8-sig"
    )
    print("\n--- Few-Shot Benchmark Results ---")
    print(pd.DataFrame([results_row]).to_string(index=False))


if __name__ == "__main__":
    main()